# Roboflow -> Experiment1

In [16]:
import random
import shutil
from pathlib import Path

In [17]:
OUT = Path("Experiment1")

SRC = Path("Selected65") 
SRC_IMG = SRC / "images" 
SRC_LBL = SRC / "labels"

# Hard reset output folder
if OUT.exists():
    shutil.rmtree(OUT)

# Recreate structure
TRAIN_IMG = OUT / "train/images"
TRAIN_LBL = OUT / "train/labels"
VAL_IMG = OUT / "val/images"
VAL_LBL = OUT / "val/labels"

for p in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    p.mkdir(parents=True)


In [18]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

QUEEN_ID_SRC = 3   # queen index
QUEEN_ID_NEW = 0   # make it single-class

def filter_to_queen(label_path: Path) -> str:
    """
    Returns new label text containing only queen boxes, remapped to class 0.
    If no queen boxes, returns "" (empty).
    """
    lines_out = []
    for line in label_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        cls = int(float(parts[0]))  # robust if "3.0"
        if cls == QUEEN_ID_SRC:
            parts[0] = str(QUEEN_ID_NEW)
            lines_out.append(" ".join(parts))
    return "\n".join(lines_out) + ("\n" if lines_out else "")

pairs = []
missing_lbl = 0

for img in SRC_IMG.iterdir():
    if img.suffix.lower() not in IMG_EXTS:
        continue
    lbl = SRC_LBL / (img.stem + ".txt")
    if not lbl.exists():
        missing_lbl += 1
        continue
    pairs.append((img, lbl))

print(f"Found pairs: {len(pairs)}  | Missing labels: {missing_lbl}")


Found pairs: 65  | Missing labels: 0


In [19]:
random.seed(42)
random.shuffle(pairs)

split_idx = int(0.8 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

print("Train:", len(train_pairs))
print("Val:  ", len(val_pairs))


Train: 52
Val:   13


In [20]:
def write_split_include_negatives(pairs, img_out: Path, lbl_out: Path):
    pos = 0
    neg = 0
    for img, lbl in pairs:
        new_txt = filter_to_queen(lbl)  # queen-only, remapped to class 0

        # Copy image always
        shutil.copy2(img, img_out / img.name)

        # Write label always (empty file = negative example)
        out_lbl = lbl_out / (img.stem + ".txt")
        out_lbl.write_text(new_txt)  # may be "" (empty)

        if new_txt.strip():
            pos += 1
        else:
            neg += 1

    return pos, neg

pos_tr, neg_tr = write_split_include_negatives(train_pairs, TRAIN_IMG, TRAIN_LBL)
pos_va, neg_va = write_split_include_negatives(val_pairs, VAL_IMG, VAL_LBL)

print(f"✅ Train positives (has queen): {pos_tr}, negatives (no queen): {neg_tr}")
print(f"✅ Val   positives (has queen): {pos_va}, negatives (no queen): {neg_va}")


✅ Train positives (has queen): 25, negatives (no queen): 27
✅ Val   positives (has queen): 7, negatives (no queen): 6


In [21]:
def sanity_report(img_dir: Path, lbl_dir: Path, name: str):
    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png"}])
    lbls = sorted([p for p in lbl_dir.iterdir() if p.suffix.lower() == ".txt"])

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    missing_lbl = sorted(img_stems - lbl_stems)
    missing_img = sorted(lbl_stems - img_stems)

    empty_lbl = 0
    nonempty_lbl = 0
    for p in lbls:
        txt = p.read_text().strip()
        if txt == "":
            empty_lbl += 1
        else:
            nonempty_lbl += 1

    print(f"\n--- {name} ---")
    print(f"Images: {len(imgs)}")
    print(f"Labels: {len(lbls)}")
    print(f"Non-empty labels (queen present): {nonempty_lbl}")
    print(f"Empty labels (no queen): {empty_lbl}")

    if missing_lbl:
        print(f"⚠️ Images missing labels: {len(missing_lbl)} (showing up to 10)")
        print(missing_lbl[:10])
    else:
        print("✅ Every image has a label file")

    if missing_img:
        print(f"⚠️ Labels missing images: {len(missing_img)} (showing up to 10)")
        print(missing_img[:10])
    else:
        print("✅ Every label corresponds to an image")

sanity_report(TRAIN_IMG, TRAIN_LBL, "Experiment1 / train")
sanity_report(VAL_IMG, VAL_LBL, "Experiment1 / val")



--- Experiment1 / train ---
Images: 52
Labels: 52
Non-empty labels (queen present): 25
Empty labels (no queen): 27
✅ Every image has a label file
✅ Every label corresponds to an image

--- Experiment1 / val ---
Images: 13
Labels: 13
Non-empty labels (queen present): 7
Empty labels (no queen): 6
✅ Every image has a label file
✅ Every label corresponds to an image
